# AKD-EXT Closed Loop CM1 Playground


## Closed loop workflow - CM1

A pipeline of 4 agents for end-to-end research workflows. These agents are designed to work sequentially, where each stage's output feeds into the next.

**Required env vars:** `OPENAI_API_KEY`

| Stage | Agent | Purpose |
|-------|-------|---------|
| 2 | CM1CapabilityFeasibilityMapperAgent | Assesses feasibility of a research question |
| 3 | CM1WorkflowSpecBuilderAgent | Builds experiment workflow spec from hypotheses |
| 4 | CM1ExperimentImplementationAgent | Creates implementation workspace (SLURM, configs) |
| 5 | CM1ResearchReportGeneratorAgent | Generates analysis notebooks and reports |

### Stage 1: Gap Agent Output

In [1]:
# Stage 1: Gap Agent Output

research_question = """# Environmental Stability Control via Input Sounding

## Research Question

**How does atmospheric stability in the environmental temperature profile (stable vs unstable input sounding) influence tropical cyclone intensity and structural evolution?**

---

## Problem Framing

Many idealized tropical cyclone simulations assume a single environmental thermodynamic profile, but the vertical temperature stability of the environment has a strong influence on convection organization, boundary-layer overturning, and eyewall dynamics. Most previous work has focused on surface fluxes or drag processes, with the role of background thermodynamic stability (controlled via the input sounding) rarely isolated.

Understanding whether instability-driven convection enhances storm intensification — compared to stable stratification that suppresses vertical motion — can help clarify the thermodynamic constraints on tropical cyclone development.

- **Origin:** Research design extension
- **Confidence:** Moderate
- **Rationale:** Environmental stability directly controls convective buoyancy, vertical mass flux, and eyewall convection strength — key drivers of tropical cyclone intensity.

---

## Evidence Anchor (Intra-Corpus)

- **Convective instability and tropical cyclone intensification:** Emanuel (1986, 1995)
- **Environmental thermodynamic control of convection:** Rotunno & Emanuel (1987)
- **Role of environmental stability in eyewall convection and storm structure:** Montgomery & Smith (2014)

---

## Variables / Diagnostics

- Maximum wind speed evolution
- Minimum central pressure
- Convective intensity and vertical velocity distribution
- Eyewall structure and radius of maximum wind (RMW)
- CAPE / atmospheric stability metrics derived from sounding
- Boundary layer thermodynamic structure
- Eyewall updraft magnitude and organization

---

## Causality Guardrails

- Stability changes **must only be introduced** through temperature profile modifications in the input sounding.
- Surface fluxes and drag formulations **must remain constant** to isolate thermodynamic effects.
- Interpretation should focus on structural and dynamical response before attributing causal mechanisms.

---

## Hypothesis

**Hypothesis 3.1:**
Tropical cyclone intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.

---

## Why This Remains Secondary

Environmental stability interacts strongly with other thermodynamic controls such as surface fluxes, moisture availability, and drag processes. Without isolating these mechanisms first, interpretation of stability-driven intensity changes may be confounded by coupled processes.
"""

### Stage 2: Capability/Feasibility Mapper Agent

In [2]:
from akd_ext.agents.closed_loop.cm1 import (
    CM1CapabilityFeasibilityMapperAgent,
    CM1CapabilityFeasibilityMapperConfig,
)
from akd_ext.agents.closed_loop.stages.capability_feasibility_mapper import (
    CapabilityFeasibilityMapperInputSchema,
)

/Users/rsahoo/Desktop/AKD-EXT-Latest/akd-ext/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# Stage 2: Capability Feasibility Mapper
mapper_agent = CM1CapabilityFeasibilityMapperAgent(CM1CapabilityFeasibilityMapperConfig())

result = await mapper_agent.arun(
    CapabilityFeasibilityMapperInputSchema(
        research_questions_md=research_question,
    )
)

print(result.report)

# CARE Capability & Feasibility Assessment — Environmental Stability Control via Input Sounding (CM1)

This report provides capability/feasibility assessments and evidence paths only. It does not make a final decision to run experiments; human approval is required.

---

## Step 1 — Validate Inputs

### Inputs present
- Research questions + hypothesis (provided in `research_questions_md`).
- CM1 case configurations and example namelists for idealized hurricanes and other test cases.
  - Evidence: `/run/config_files/hurricane_axisymmetric/README`, `/run/config_files/hurricane_axisymmetric/namelist.input`, `/run/config_files/hurricane_axisymmetric/input_sounding`
  - Evidence: `/run/config_files/hurricane_3d_cpm/README`, `/run/config_files/hurricane_3d_cpm/namelist.input`, `/run/config_files/hurricane_3d_cpm/input_sounding`
- Cluster policy/limits (UAH Matrix) available.
  - Evidence: `UAH Matrix IT wiki: Matrix.html#Queues` (provided as `cluster_it_context`)

### Critical missing / not 

In [4]:
# Saving the md file for testing purpose
from pathlib import Path

report_md_path = Path("result_report.md")
report_md_path.write_text(result.report, encoding="utf-8")
print(f"Saved report to: {report_md_path}")
print(result.report)

Saved report to: result_report.md
# CARE Capability & Feasibility Assessment — Environmental Stability Control via Input Sounding (CM1)

This report provides capability/feasibility assessments and evidence paths only. It does not make a final decision to run experiments; human approval is required.

---

## Step 1 — Validate Inputs

### Inputs present
- Research questions + hypothesis (provided in `research_questions_md`).
- CM1 case configurations and example namelists for idealized hurricanes and other test cases.
  - Evidence: `/run/config_files/hurricane_axisymmetric/README`, `/run/config_files/hurricane_axisymmetric/namelist.input`, `/run/config_files/hurricane_axisymmetric/input_sounding`
  - Evidence: `/run/config_files/hurricane_3d_cpm/README`, `/run/config_files/hurricane_3d_cpm/namelist.input`, `/run/config_files/hurricane_3d_cpm/input_sounding`
- Cluster policy/limits (UAH Matrix) available.
  - Evidence: `UAH Matrix IT wiki: Matrix.html#Queues` (provided as `cluster_it_cont

In [5]:
# read the md file for testing purpose
from pathlib import Path

report_md_path = Path("result_report.md")
# report_md_path.write_text(report_md_content, encoding="utf-8")
report_md_content = report_md_path.read_text(encoding="utf-8")

### Stage 3: Workflow Spec Builder

In [6]:
from akd_ext.agents.closed_loop.cm1 import (
    CM1WorkflowSpecBuilderAgent,
    CM1WorkflowSpecBuilderConfig,
)
from akd_ext.agents.closed_loop.stages.workflow_spec_builder import (
    WorkflowSpecBuilderInputSchema,
)

In [27]:
# Stage 3: Workflow Spec Builder
# Takes Gap Agent output (hypotheses) + Stage 2 feasibility report
spec_agent = CM1WorkflowSpecBuilderAgent(CM1WorkflowSpecBuilderConfig())

spec_result = await spec_agent.arun(
    WorkflowSpecBuilderInputSchema(
        stage_1_hypotheses=research_question,
        stage_2_feasibility=report_md_content,
    )
)

print("SPEC:")
print("=" * 80)
print(spec_result.spec)
print("\nREASONING:")
print("=" * 80)
print(spec_result.reasoning)

SPEC:
# Metadata
status: draft
model: CM1
workflow_name: Environmental stability control via input_sounding (tropical cyclone)
primary_rq_tag_or_rq_id: RQ-001
hypotheses_in_scope: Hypothesis 3.1
baseline_experiment_id: EXP_STAB_baseline
planned_experiment_count: 3

## Sources
- cpm_RadConvEquil/namelist.input
- hurricane_3d_cpm/input_sounding
- hurricane_3d_cpm/namelist.input

# Control Definition
## Baseline concept
Use the **CM1 idealized 3D hurricane CPM** configuration as the control storm, and isolate **environmental stability** effects by changing **only the environmental temperature (potential temperature) profile** in the `input_sounding` file.

## Baseline inputs to start from
- Baseline case template: `hurricane_3d_cpm/namelist.input`
- Baseline environmental sounding file: `hurricane_3d_cpm/input_sounding`

## Baseline invariants (must be identical across all experiments)
- Same hurricane case configuration (domain, numerics, physics), including but not limited to:
  - file-

In [28]:
# Save the workflow spec as a markdown file
spec_md_path = Path("workflow_spec.md")
spec_md_path.write_text(spec_result.spec + "\n\n---\n\n# Reasoning\n\n" + spec_result.reasoning, encoding="utf-8")
print(f"Saved workflow spec to: {spec_md_path}")

Saved workflow spec to: workflow_spec.md


In [29]:
from pathlib import Path

workflow_spec_path = Path("workflow_spec.md")
workflow_spec_content = workflow_spec_path.read_text(encoding="utf-8")

### Stage 4 A: Experiment Implementation Agent - LLM Part to generate the JSON

In [7]:
from akd_ext.agents.closed_loop.cm1 import (
    CM1ExperimentImplementationAgent,
    CM1ExperimentImplementationConfig,
)
from akd_ext.agents.closed_loop.stages.experiment_implementation import (
    ExperimentImplementationInputSchema,
    ExperimentImplementationOutputSchema,
)

In [31]:
# Stage 4A: Experiment Implementation Planner
# GPT generates experiments, calls job_submit MCP, returns job_id + report
impl_agent = CM1ExperimentImplementationAgent(CM1ExperimentImplementationConfig())

impl_result = await impl_agent.arun(ExperimentImplementationInputSchema(stage_3_spec=workflow_spec_content))

if isinstance(impl_result, ExperimentImplementationOutputSchema):
    print(f"Job ID: {impl_result.job_id}")
    print()
    print("REPORT:")
    print("=" * 80)
    print(impl_result.report)
else:
    print(impl_result.content)

Job ID: 9f704775

REPORT:
## 1. Job submission
- **job_id:** `9f704775`
- **workspace_name:** `cm1_hurr_stability_sounding`
- **base_template:** `hurricane_3d_cpm`

## 2. Experiments implemented (3 total)

### EXP_STAB_baseline (baseline)
**Edits applied**
- `namelist.input` (group `param9`):
  - `output_cape = 1`
  - `output_cin = 1`
  - `output_lcl = 1`
  - `output_lfc = 1`

**Evidence for parameter names/groups**: `run/config_files/hurricane_3d_cpm/namelist.input`

### EXP_STAB_001 (stabilized sounding)
Inherits all baseline namelist diagnostic edits **plus**:
- `input_sounding`: **file_replace** with theta-only stabilization per Stage 3 formula (Δθ ramps to **+3 K by 10 km**, capped above), leaving the **surface line unchanged**.

**Evidence for sounding file**: `run/config_files/hurricane_3d_cpm/input_sounding`

### EXP_STAB_002 (destabilized sounding)
Inherits all baseline namelist diagnostic edits **plus**:
- `input_sounding`: **file_replace** with theta-only destabilization per

In [32]:
# Save the implementation report
if isinstance(impl_result, ExperimentImplementationOutputSchema):
    impl_md_path = Path("experiment_implementation.md")
    impl_md_path.write_text(impl_result.report, encoding="utf-8")
    print(f"Saved implementation report to: {impl_md_path}")
    print(f"Job ID: {impl_result.job_id}")

Saved implementation report to: experiment_implementation.md
Job ID: 9f704775


### Stage 5: Research Report Generator

Takes the Stage 3 workflow spec + experiment IDs from Stage 4A.
The agent checks experiment status and fetches figure URLs via MCP tools,
then generates a publication-style scientific report.

**Input:** Workflow spec + experiment IDs (from Stage 4A)

**Output:** Markdown report with Abstract, Introduction, Methodology, Results, Discussion, Conclusion

**Note:** Without the MCP server, use `tools=[]` and override the prompt to skip status/figure checks.

In [8]:
from akd_ext.agents.closed_loop.cm1 import (
    CM1ResearchReportGeneratorAgent,
    CM1ResearchReportGeneratorConfig,
)
from akd_ext.agents.closed_loop.stages.research_report_generator import (
    ResearchReportGeneratorInputSchema,
    ResearchReportGeneratorOutputSchema,
)

In [34]:
experiment_ids = "EXP_stability_baseline,EXP_stability_001,EXP_stability_002"

In [35]:
# Stage 5: Research Report Generator
# Uses job_id from Stage 4A to check status and fetch figures via MCP
job_id = impl_result.job_id
print(f"Job ID: {job_id}")

report_agent = CM1ResearchReportGeneratorAgent(CM1ResearchReportGeneratorConfig())

report_result = await report_agent.arun(
    ResearchReportGeneratorInputSchema(
        workflow_spec=workflow_spec_content,
        job_id=job_id,
    ),
)

print("RESEARCH REPORT:")
print("=" * 80)
if isinstance(report_result, ResearchReportGeneratorOutputSchema):
    print(report_result.report)
else:
    print(report_result.content)

Job ID: 9f704775
RESEARCH REPORT:
## 1. Abstract
Environmental thermodynamic stability is hypothesized to modulate tropical cyclone (TC) buoyancy, eyewall convection, and thus intensification and peak intensity. A set of CM1 idealized 3D hurricane cloud-permitting simulations was conducted in which environmental stability was isolated by applying temperature-only perturbations to the file-based input sounding while holding moisture, winds, surface exchange, and vortex initialization fixed. The provided diagnostics and summary plots permit qualitative comparison of intensity evolution, pressure evolution, structural proxies, and energetics among a baseline and paired stability perturbations. The results are consistent with stability-dependent modulation of convective response and storm evolution, but all scientific interpretations remain **(pending researcher validation)**.

*This report was generated with AI assistance and requires researcher validation before publication.*

## 2. Intr

In [36]:
# Save the research report
if isinstance(report_result, ResearchReportGeneratorOutputSchema):
    research_report_path = Path("research_report.md")
    research_report_path.write_text(report_result.report, encoding="utf-8")
    print(f"Saved research report to: {research_report_path}")
else:
    print("No structured report to save (agent returned TextOutput)")

Saved research report to: research_report.md
